# [13-1강] Sequence Data와 Hidden State - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. sequence batch 만들기

시계열/문장처럼 순서가 있는 데이터를 `[batch, seq_len, input_size]` 형태로 만듭니다.

In [4]:
batch_size, seq_len, input_size = 4, 6, 3
# TODO: [batch, seq_len, input_size] 형태의 입력을 만드세요.
x = torch.randn(batch_size, seq_len, input_size)
print('x shape:', x.shape)


x shape: torch.Size([4, 6, 3])


## 문제 2. RNN output과 hidden shape 확인하기

nn.RNN에 sequence batch를 넣고 전체 time step output과 마지막 hidden state shape를 확인합니다.

In [6]:
x = torch.randn(4, 6, 3)
# TODO: input_size=3, hidden_size=5인 RNN을 만드세요.
rnn = nn.RNN(input_size=3, hidden_size=5, batch_first=True)
try:
    output, h_n = rnn(x)
    print('output:', output.shape)
    print('h_n:', h_n.shape)
except RuntimeError as e:
    print('RNN 입력 크기를 다시 확인하세요:', str(e).split('\\n')[0])


output: torch.Size([4, 6, 5])
h_n: torch.Size([1, 4, 5])


### 해설 및 실행 결과 해석

- output은 모든 time step의 hidden state를 담고, h_n은 마지막 time step의 hidden state를 layer별로 담습니다. sequence 분류에서는 보통 마지막 hidden을 사용합니다.

## 문제 3. 마지막 hidden으로 sequence 분류하기

RNN의 마지막 hidden state를 Linear classifier에 연결해 sequence label을 예측합니다.

In [7]:
x = torch.randn(5, 7, 4)
rnn = nn.RNN(input_size=4, hidden_size=6, batch_first=True)
classifier = nn.Linear(6, 2)
output, h_n = rnn(x)
# TODO: 마지막 layer hidden을 꺼내 classifier에 넣으세요.
last_hidden = h_n[-1]
logits = classifier(last_hidden)
print(logits.shape)


torch.Size([5, 2])


### 해설 및 실행 결과 해석

- `h_n[-1]`은 마지막 layer의 최종 hidden state입니다. 이 값은 sequence 전체를 요약한 표현으로 볼 수 있어 sequence classification의 입력으로 자주 사용됩니다.